# Assignment Beta — Question 3: Knowledge Distillation (worked calculation)
**Course:** Agentic AI (AI4015), FAST NUCES Karachi  **Student ID:** 22K-8706

Prompt: *"The patient was admitted to the ___"*; vocabulary = [hospital, clinic, ward, school], gold label `hospital` → y = [1,0,0,0].

Teacher logits $z_t=[5,3,2,0]$, student logits $z_s=[3,2.5,1,0.5]$, $T=2$, $\alpha=0.5$, natural log.

$$L=\alpha\,T^2\,\mathrm{KL}(p^T\Vert q^T)+(1-\alpha)\,\mathrm{CE}(y,q^1)$$

In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)
words = ["hospital", "clinic", "ward", "school"]
zt = np.array([5.0, 3.0, 2.0, 0.0])
zs = np.array([3.0, 2.5, 1.0, 0.5])
y  = np.array([1.0, 0.0, 0.0, 0.0])
T, alpha, lr = 2.0, 0.5, 0.5

def softmax(z):
    e = np.exp(z - z.max())          # subtract max for numerical stability
    return e / e.sum()

def show(name, v):
    print(f"{name:>10}: " + "  ".join(f"{w}={x:.4f}" for w, x in zip(words, v)))

## (a) Probabilities at T = 1
$p_i=\dfrac{e^{z_i}}{\sum_j e^{z_j}}$

In [2]:
p1 = softmax(zt); q1 = softmax(zs)
print("teacher exp(z):", np.exp(zt), " sum =", np.exp(zt).sum())
print("student exp(z):", np.exp(zs), " sum =", np.exp(zs).sum())
show("teacher", p1); show("student", q1)

teacher exp(z): [148.413159  20.085537   7.389056   1.      ]  sum = 176.8877521246949
student exp(z): [20.085537 12.182494  2.718282  1.648721]  sum = 36.63503398305032
   teacher: hospital=0.8390  clinic=0.1135  ward=0.0418  school=0.0057
   student: hospital=0.5483  clinic=0.3325  ward=0.0742  school=0.0450


## (b) Softened probabilities at T = 2
$p^T_i=\mathrm{softmax}(z_i/T)$ — dividing by T flattens the distribution and exposes the teacher's "dark knowledge" (clinic/ward are plausible, school is not).

In [3]:
pT = softmax(zt / T); qT = softmax(zs / T)
print("teacher z/T:", zt / T, " student z/T:", zs / T)
show("teacher T=2", pT); show("student T=2", qT)

teacher z/T: [2.5 1.5 1.  0. ]  student z/T: [1.5  1.25 0.5  0.25]
teacher T=2: hospital=0.5977  clinic=0.2199  ward=0.1334  school=0.0491
student T=2: hospital=0.4110  clinic=0.3201  ward=0.1512  school=0.1177


## (c) KL divergence  $\mathrm{KL}(p^T\Vert q^T)=\sum_i p^T_i\ln\dfrac{p^T_i}{q^T_i}$

In [4]:
terms = pT * np.log(pT / qT)
for w, a, b, t in zip(words, pT, qT, terms):
    print(f"{w:>9}: {a:.4f} * ln({a:.4f}/{b:.4f}) = {t:+.6f}")
KL = terms.sum(); print(f"KL = {KL:.6f} nats")

 hospital: 0.5977 * ln(0.5977/0.4110) = +0.223852
   clinic: 0.2199 * ln(0.2199/0.3201) = -0.082559
     ward: 0.1334 * ln(0.1334/0.1512) = -0.016734
   school: 0.0491 * ln(0.0491/0.1177) = -0.042952
KL = 0.081607 nats


## (d) Hard-label cross-entropy  $\mathrm{CE}=-\sum_i y_i\ln q^1_i=-\ln q^1_{hospital}$

In [5]:
CE = -np.sum(y * np.log(q1)); print(f"CE = -ln({q1[0]:.4f}) = {CE:.6f}")

CE = -ln(0.5483) = 0.601005


## (e) Total distillation loss

In [6]:
soft_term = alpha * T**2 * KL
hard_term = (1 - alpha) * CE
L = soft_term + hard_term
print(f"soft = 0.5*4*{KL:.6f} = {soft_term:.6f}")
print(f"hard = 0.5*{CE:.6f}   = {hard_term:.6f}")
print(f"L    = {L:.6f}")

soft = 0.5*4*0.081607 = 0.163215
hard = 0.5*0.601005   = 0.300502
L    = 0.463717


## (f) Gradient w.r.t. the student logits
Using $\partial\,\mathrm{KL}(p^T\Vert q^T)/\partial z_{s,i}=\tfrac1T(q^T_i-p^T_i)$ and $\partial\,\mathrm{CE}/\partial z_{s,i}=q^1_i-y_i$:

$$\frac{\partial L}{\partial z_{s,i}}=\alpha T\,(q^T_i-p^T_i)+(1-\alpha)(q^1_i-y_i)$$
(The $T^2$ factor cancels one $1/T$, which is exactly why Hinton et al. scale the soft loss by $T^2$.)

In [7]:
g_soft = alpha * T * (qT - pT)
g_hard = (1 - alpha) * (q1 - y)
grad = g_soft + g_hard
show("soft grad", g_soft); show("hard grad", g_hard); show("total", grad)
print("sum of gradient =", round(grad.sum(), 12), "(softmax gradients always sum to 0)")

# finite-difference check
def loss_fn(z):
    pT_, qT_, q1_ = softmax(zt/T), softmax(z/T), softmax(z)
    return alpha*T**2*np.sum(pT_*np.log(pT_/qT_)) + (1-alpha)*(-np.log(q1_[0]))
eps = 1e-6
num = np.array([(loss_fn(zs+eps*e) - loss_fn(zs-eps*e)) / (2*eps) for e in np.eye(4)])
print("max |analytic - numeric| =", np.abs(num - grad).max())

 soft grad: hospital=-0.1867  clinic=0.1002  ward=0.0178  school=0.0687
 hard grad: hospital=-0.2259  clinic=0.1663  ward=0.0371  school=0.0225
     total: hospital=-0.4126  clinic=0.2665  ward=0.0549  school=0.0912
sum of gradient = 0.0 (softmax gradients always sum to 0)
max |analytic - numeric| = 2.380221297837437e-10


## (g) One gradient-descent step (η = 0.5) and the new loss
$z_s' = z_s - \eta\,\nabla_{z_s}L$

In [8]:
zs_new = zs - lr * grad
show("new logits", zs_new)
pT2, qT2, q12 = softmax(zt/T), softmax(zs_new/T), softmax(zs_new)
KL2 = np.sum(pT2*np.log(pT2/qT2)); CE2 = -np.log(q12[0]); L2 = alpha*T**2*KL2 + (1-alpha)*CE2
show("q (T=1)", q12); show("q (T=2)", qT2)
print(f"KL' = {KL2:.6f}, CE' = {CE2:.6f}, L' = {L2:.6f}")
print(f"Loss decreased by {L - L2:.6f} ({(L - L2)/L*100:.1f}%)")

new logits: hospital=3.2063  clinic=2.3668  ward=0.9725  school=0.4544
   q (T=1): hospital=0.6239  clinic=0.2695  ward=0.0668  school=0.0398
   q (T=2): hospital=0.4470  clinic=0.2938  ward=0.1463  school=0.1129
KL' = 0.056682, CE' = 0.471783, L' = 0.349256
Loss decreased by 0.114461 (24.7%)


## Interpretation
* The student moves `hospital` up and pushes `clinic`, `ward` and especially `school` down, i.e. towards the teacher's shape **and** the gold label.
* `school` gets a positive gradient mainly from the soft term: the student gives it 11.8% at T=2 whereas the teacher gives only 4.9%, so distillation teaches the student *which wrong answers are very wrong*, which the one-hot label alone cannot.
* After one step the loss drops from ≈0.4637 to ≈0.3493 and the KL term falls by about 30%.